# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.727650714929437)),
 (1, np.float64(2.727650714929437)),
 (2, np.float64(2.727650714929437)),
 (3, np.float64(2.727650714929437)),
 (4, np.float64(2.727650714929437))]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [('a', 2)])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.0233506857615583)),
   (None, np.float64(0.046716421134139985)),
   (None, np.float64(0.06532875372664204)),
   (None, np.float64(0.12501239397895125)),
   (None, np.float64(0.13907935546020822)),
   (None, np.float64(0.1440398495512203)),
   (None, np.float64(0.2073771643467388)),
   (None, np.float64(0.22256781079353383)),
   (None, np.float64(0.24334509062709198)),
   (None, np.float64(0.2643341271234664)),
   (None, np.float64(0.2702308736107679)),
   (None, np.float64(0.3131582538866692)),
   (None, np.float64(0.3612931559763014)),
   (None, np.float64(0.3791062381446564)),
   (None, np.float64(0.42774580514478844)),
   (None, np.float64(0.43771417568253557)),
   (None, np.float64(0.46912568822939826)),
   (None, np.float64(0.4795113825213104)),
   (None, np.float64(0.48640084760349966))]),
 (1,
  [(None, np.float64(0.5518534298463291)),
   (None, np.float64(0.579579959442894)),
   (None, np.float64(0.5906943937470364)),
   (None, np.float64(0.609664335

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
import numpy as np

input_collection = np.random.randint(0, 1000, 50).tolist()

def RECORDREADER():
  return [(None, x) for x in input_collection]

def MAP(_, val):
  yield ("max", val)

def REDUCE(key, values):
  yield (key, max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Input max: {max(input_collection)}")
print(f"MapReduce max: {output}")

Input max: 966
MapReduce max: [('max', 966)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
input_collection = [10, 20, 30, 40, 50]

def RECORDREADER():
  return [(None, x) for x in input_collection]

def MAP(_, val):
  yield ("avg", (val, 1))

def REDUCE(key, values):
  total_sum = 0
  total_count = 0
  for val, count in values:
    total_sum += val
    total_count += count
  yield (key, total_sum / total_count)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Input mean: {np.mean(input_collection)}")
print(f"MapReduce mean: {output}")

Input mean: 30.0
MapReduce mean: [('avg', 30.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def groupbykey_sort(iterable):
  # Сортируем по ключу (k2)
  sorted_iterable = sorted(iterable, key=lambda x: x[0])
  if not sorted_iterable:
    return

  current_key, current_values = sorted_iterable[0][0], [sorted_iterable[0][1]]

  for k, v in sorted_iterable[1:]:
    if k == current_key:
      current_values.append(v)
    else:
      yield (current_key, current_values)
      current_key, current_values = k, [v]
  yield (current_key, current_values)

# Тест
test_data = [('b', 1), ('a', 2), ('b', 3), ('a', 4)]
print(f"Original groupbykey: {list(groupbykey(test_data))}")
print(f"Sort-based groupbykey: {list(groupbykey_sort(test_data))}")

Original groupbykey: [('b', [1, 3]), ('a', [2, 4])]
Sort-based groupbykey: [('a', [2, 4]), ('b', [1, 3])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
input_collection = [1, 2, 2, 3, 4, 4, 4, 5]

def RECORDREADER():
  return [(x, x) for x in input_collection]

def MAP(key, val):
  yield (key, None)

def REDUCE(key, values):
  yield key

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Unique elements: {output}")

Unique elements: [1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
# Selection
users_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def RECORDREADER():
    return [(u.id, u) for u in users_collection]

def MAP(id, row):
    if row.age > 30:
        yield (row, row)

def REDUCE(key, values):
    yield key

print("Selection (age > 30):")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Selection (age > 30):
[User(id=0, age=55, social_contacts=20, gender='male'), User(id=3, age=33, social_contacts=800, gender='female')]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
# Projection (only age)
def RECORDREADER():
    return [(u.id, u) for u in users_collection]

def MAP(id, row):
    yield (row.age, row.age)

def REDUCE(key, values):
    yield key

print("Projection (age):")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Projection (age):
[55, 25, 33]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
# Union
S1 = [1, 2, 3]
S2 = [3, 4, 5]

def RECORDREADER():
    return [(x, x) for x in S1] + [(x, x) for x in S2]

def MAP(key, val):
    yield (key, key)

def REDUCE(key, values):
    yield key

print("Union:")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Union:
[1, 2, 3, 4, 5]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
# Intersection
def REDUCE(key, values):
    if len(values) > 1:
        yield key

print("Intersection:")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Intersection:
[3]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
# Difference (S1 - S2)
def RECORDREADER():
    return [(x, 'R') for x in S1] + [(x, 'S') for x in S2]

def MAP(val, source):
    yield (val, source)

def REDUCE(key, values):
    if values == ['R']:
        yield key

print("Difference (S1 - S2):")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Difference (S1 - S2):
[1, 2]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
# Natural Join
# R(A, B) and S(B, C)
R = [(1, 'x'), (2, 'y'), (3, 'x')]
S = [('x', 'apple'), ('y', 'banana'), ('z', 'cherry')]

def RECORDREADER():
    return [((r[1], 'R'), r[0]) for r in R] + [((s[0], 'S'), s[1]) for s in S]

def MAP(key_source, val):
    key, source = key_source
    yield (key, (source, val))

def REDUCE(key, values):
    r_vals = [v for source, v in values if source == 'R']
    s_vals = [v for source, v in values if source == 'S']
    for r in r_vals:
        for s in s_vals:
            yield (r, key, s)

print("Natural Join (R join S on B):")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Natural Join (R join S on B):
[(1, 'x', 'apple'), (3, 'x', 'apple'), (2, 'y', 'banana')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
# Grouping and Aggregation
def RECORDREADER():
    return [(u.id, u) for u in users_collection]

def MAP(id, row):
    # Group by gender, aggregate social_contacts
    yield (row.gender, row.social_contacts)

def REDUCE(gender, contacts_list):
    yield (gender, sum(contacts_list))

print("Grouping by gender (Sum of social contacts):")
print(list(MapReduce(RECORDREADER, MAP, REDUCE)))

Grouping by gender (Sum of social contacts):
[('male', 20), ('female', 1540)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
# Matrix-Vector multiplication (Vector does not fit in memory)

def RECORDREADER():
    # Matrix M: (i, j, v)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            yield (j, ('M', i, mat[i, j]))
    # Vector V: (j, w)
    for j in range(len(vec)):
        yield (j, ('V', vec[j]))

def MAP(j, val_tuple):
    # Просто передаем дальше для группировки по j
    yield (j, val_tuple)

def REDUCE(j, values):
    # Сначала извлекаем значение вектора
    w = None
    m_entries = []
    for item in values:
        if item[0] == 'V':
            w = item[1]
        else:
            m_entries.append((item[1], item[2])) # (i, m_val)

    if w is not None:
        for i, m_val in m_entries:
            yield (i, m_val * w)

# Первый проход
products = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER_STEP2():
    return products

def MAP_STEP2(i, product):
    yield (i, product)

def REDUCE_STEP2(i, products_list):
    yield (i, sum(products_list))

# Второй проход
final_vector = list(MapReduce(RECORDREADER_STEP2, MAP_STEP2, REDUCE_STEP2))

print("Matrix-Vector Result (Streaming):")
# Сортируем по индексу строки для вывода
final_sorted = sorted(final_vector, key=lambda x: x[0])
print(final_sorted)

# Проверка
ref = np.dot(mat, vec)
print(f"Correct: {np.allclose([v for i, v in final_sorted], ref)}")

Matrix-Vector Result (Streaming):
[(0, np.float64(2.727650714929437)), (1, np.float64(2.727650714929437)), (2, np.float64(2.727650714929437)), (3, np.float64(2.727650714929437)), (4, np.float64(2.727650714929437))]
Correct: True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 40
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(I):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  yield (key, sum(values))

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = list(MapReduce(RECORDREADER, MAP, REDUCE))

def asmatrix(reduce_output):
  res_mat = np.zeros((I, K))
  for ((i, k), val) in reduce_output:
    res_mat[i, k] = val
  return res_mat

print(f"Result matches: {np.allclose(reference_solution, asmatrix(solution))}")

Result matches: True


In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [ ]:
# Обе матрицы генерируются в RECORDREADER
def RECORDREADER():
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            yield (('M', i, j), small_mat[i, j])
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield (('N', j, k), big_mat[j, k])

def MAP(key, val):
    if key[0] == 'M':
        _, i, j = key
        yield (j, ('M', i, val))
    else:
        _, j, k = key
        yield (j, ('N', k, val))

def REDUCE(j, values):
    m_elements = [(i, v) for tag, i, v in values if tag == 'M']
    n_elements = [(k, w) for tag, k, w in values if tag == 'N']
    for i, v in m_elements:
        for k, w in n_elements:
            yield ((i, k), v * w)

join_output = list(MapReduce(RECORDREADER, MAP, REDUCE))

def MAP2(ik, product):
    yield (ik, product)

def REDUCE2(ik, products):
    yield (ik, sum(products))

def RECORDREADER2():
    return join_output

final_output = list(MapReduce(RECORDREADER2, MAP2, REDUCE2))
print(f"Two-pass MR Result matches: {np.allclose(reference_solution, asmatrix(final_output))}")

Two-pass MR Result matches: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [ ]:
# Distributed Matrix Multiplication
reducers = 4

def INPUTFORMAT():
    def reader_m():
        for i in range(small_mat.shape[0]):
            for j in range(small_mat.shape[1]):
                yield (('M', i, j), small_mat[i, j])
    def reader_n():
        for j in range(big_mat.shape[0]):
            for k in range(big_mat.shape[1]):
                yield (('N', j, k), big_mat[j, k])
    yield reader_m()
    yield reader_n()

partitioned_join = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)
join_res = []
for pid, p_iter in partitioned_join:
    join_res.extend(list(p_iter))

def INPUTFORMAT2():
    yield join_res

final_partitioned = MapReduceDistributed(INPUTFORMAT2, MAP2, REDUCE2)
final_res = []
for pid, p_iter in final_partitioned:
    final_res.extend(list(p_iter))

print(f"Distributed Result matches: {np.allclose(reference_solution, asmatrix(final_res))}")

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.
Distributed Result matches: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [ ]:
import random

def INPUTFORMAT_GENERAL():
    # Имитируем несколько ридеров
    all_m = [('M', i, j, small_mat[i, j]) for i in range(I) for j in range(J)]
    all_n = [('N', j, k, big_mat[j, k]) for j in range(J) for k in range(K)]

    all_data = all_m + all_n
    random.shuffle(all_data)

    # Разделим на 3 части вручную, чтобы избежать приведения типов numpy
    n = len(all_data)
    size = n // 3
    splits = [all_data[i:i + size] for i in range(0, n, size)]

    for part in splits:
        yield [((tag, i_j, j_k), val) for tag, i_j, j_k, val in part]

# Повторный запуск распределенного Join
join_dist = MapReduceDistributed(INPUTFORMAT_GENERAL, MAP, REDUCE)
join_res_list = []
for _, it in join_dist: join_res_list.extend(list(it))

def INPUTFORMAT_SUM():
    yield join_res_list

final_dist = MapReduceDistributed(INPUTFORMAT_SUM, MAP2, REDUCE2)
final_res_list = []
for _, it in final_dist: final_res_list.extend(list(it))

print(f"Generalized MR matches: {np.allclose(reference_solution, asmatrix(final_res_list))}")

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.
Generalized MR matches: True
